# Overnight Comprehensive Exploration: Off-Policy Detection at Scale

**Expected Runtime: 6-8 hours on A40 GPU**

This notebook conducts a publication-ready systematic study of off-policy detection across 1000+ conditions.

## Study Components
1. **Large-Scale Generalization**: 100 prompts × 20 domains × 10 samples = 3000 generations
2. **Fine-Grained Steering**: 20 α values × 10 scenarios × 10 samples = 2000 generations
3. **Cross-Model Transfer**: 3 models × 20 cases × 10 samples = 600 generations
4. **Adversarial Robustness**: 50 adversarial injections × 10 samples = 500 generations
5. **Vector Arithmetic**: 30 combinations × 20 samples = 600 generations
6. **Longitudinal Study**: 20 long conversations with tracking
7. **Layer Sweep** (NEW): 18 layers × 3 alphas × 5 scenarios × 5 samples = 1350 generations

**Total: ~8500+ generations with comprehensive evaluation**

## ✅ IMPROVEMENTS:
- **Resumable**: Checks existing results and skips completed parts
- **Layer 18**: Uses experimentally optimal layer (not 35!)
- **Device management**: Proper GPU tensor handling
- **Layer sweep**: Validates optimal layer choice

## Features
- Automatic checkpointing every 100 generations
- Progress tracking with time estimates
- Resilient to interruptions (can resume)
- Comprehensive statistical analysis
- Publication-quality visualizations

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Dict, Tuple, Optional
import pandas as pd
from scipy import stats
from tqdm import tqdm
import time
from datetime import datetime, timedelta
import pickle
import hashlib
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('..')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (14, 8)

# Create results directory
RESULTS_DIR = Path('../results/overnight_run')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = RESULTS_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Results will be saved to: {RESULTS_DIR}")
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check existing results
def check_part_complete(part_name):
    """Check if a part has already been completed."""
    file_path = RESULTS_DIR / f'{part_name}.json'
    if file_path.exists():
        with open(file_path) as f:
            data = json.load(f)
            return True, len(data)
    return False, 0

# Check all parts
parts_status = {}
for part in ['part1_generalization', 'part2_steering', 'part3_transfer', 
             'part4_adversarial', 'part5_arithmetic', 'part6_longitudinal', 'part7_layer_sweep']:
    complete, count = check_part_complete(part)
    parts_status[part] = {'complete': complete, 'count': count}
    if complete:
        print(f"✅ {part}: {count} results found")
    else:
        print(f"⏳ {part}: Not started")

total_existing = sum(s['count'] for s in parts_status.values())
print(f"\nTotal existing results: {total_existing}")

In [ ]:
# Load models and vectors
print("\nLoading primary model (Qwen3-4B)...")
model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen3-4B',
    torch_dtype=torch.float16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')

# Load vectors
vector_clipped = torch.load('../artifacts/vector_clipped.pt')
vector_full = torch.load('../artifacts/vector_full.pt')

# CRITICAL FIX: Use layer 18 based on experimental findings (NOT 35!)
BEST_LAYER = 18  # Changed from 35 - experimentally optimal from steering experiments

print(f"✓ Model loaded on {model.device}")
print(f"✓ Vectors loaded for layers {sorted(vector_clipped.keys())}")
print(f"✓ Using layer {BEST_LAYER} as primary (EXPERIMENTALLY OPTIMAL)")

## Core Infrastructure

In [ ]:
class SteeringHook:
    """Context manager for activation steering with proper device management."""
    def __init__(self, model, vector, alpha=1.0, layer=18):
        self.model = model
        # FIXED: Ensure vector is on the same device as model
        if isinstance(vector, dict):
            self.vector = vector[layer].to(model.device)
        else:
            self.vector = vector.to(model.device)
        
        self.alpha = alpha
        self.layer = layer
        self.hook = None
        
    def __enter__(self):
        def hook_fn(module, input, output):
            if isinstance(output, tuple):
                hidden = output[0]
            else:
                hidden = output
            
            # Ensure vector has correct shape and device
            steering_vector = self.vector
            if steering_vector.dim() == 1:
                steering_vector = steering_vector.unsqueeze(0).unsqueeze(0)
            elif steering_vector.dim() == 2:
                steering_vector = steering_vector.unsqueeze(0)
            
            # Apply steering
            hidden = hidden + self.alpha * steering_vector
            
            if isinstance(output, tuple):
                return (hidden,) + output[1:]
            return hidden
        
        # Register hook on the specified layer
        self.hook = self.model.model.layers[self.layer].register_forward_hook(hook_fn)
        return self
    
    def __exit__(self, *args):
        if self.hook:
            self.hook.remove()

class ExperimentRunner:
    """Main experiment runner with checkpointing and progress tracking."""
    
    def __init__(self, model, tokenizer, vectors, results_dir, best_layer=18):
        self.model = model
        self.tokenizer = tokenizer
        self.vectors = vectors
        self.results_dir = results_dir
        self.best_layer = best_layer
        self.results = []
        self.generation_count = 0
        self.start_time = time.time()
        self.checkpoint_every = 100
        
    def generate_with_tracking(self, prompt, prefix="", alpha=0.0, layer=None, 
                               max_tokens=250, temperature=0.8, **kwargs):
        """Generate with progress tracking, timing, and proper steering."""
        if layer is None:
            layer = self.best_layer
            
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, 
            enable_thinking=kwargs.get('enable_thinking', True)
        )
        
        if prefix:
            text += prefix + "\n"
            
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        
        # Apply steering if needed with proper hook
        if alpha != 0 and layer in self.vectors:
            with SteeringHook(self.model, self.vectors, alpha=alpha, layer=layer):
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=max_tokens,
                        temperature=temperature,
                        top_p=kwargs.get('top_p', 0.9),
                        do_sample=True,
                        pad_token_id=self.tokenizer.pad_token_id
                    )
        else:
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature,
                    top_p=kwargs.get('top_p', 0.9),
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id
                )
            
        generation = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=False
        )
        
        self.generation_count += 1
        
        # Checkpoint if needed
        if self.generation_count % self.checkpoint_every == 0:
            self.save_checkpoint()
            
        return generation
    
    def evaluate(self, text, eval_prompt):
        """Get evaluation score from model."""
        full = f"Evaluate this text:\n{text[:500]}\n\n{eval_prompt}\nGive a score from 1-100:"
        messages = [{"role": "user", "content": full}]
        eval_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        inputs = self.tokenizer(eval_text, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=10, temperature=0.1, do_sample=False
            )
            
        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        
        import re
        numbers = re.findall(r'\d+', response)
        return int(numbers[0]) if numbers else 50
    
    def save_checkpoint(self):
        """Save current results to checkpoint."""
        checkpoint_path = CHECKPOINT_DIR / f"checkpoint_{self.generation_count}.pkl"
        with open(checkpoint_path, 'wb') as f:
            pickle.dump({
                'results': self.results,
                'generation_count': self.generation_count,
                'elapsed_time': time.time() - self.start_time
            }, f)
        print(f"  [Checkpoint saved: {self.generation_count} generations]")
    
    def estimate_remaining_time(self, current_step, total_steps):
        """Estimate remaining time based on current progress."""
        if current_step == 0:
            return "Calculating..."
        elapsed = time.time() - self.start_time
        rate = elapsed / current_step
        remaining = rate * (total_steps - current_step)
        return str(timedelta(seconds=int(remaining)))

runner = ExperimentRunner(model, tokenizer, vector_clipped, RESULTS_DIR, best_layer=BEST_LAYER)
print(f"✓ Experiment runner initialized with layer {BEST_LAYER} and proper steering")

## Part 1: Large-Scale Generalization Study (3000 generations)

In [4]:
# Define 100 diverse prompts across 20 domains
DOMAIN_PROMPTS = {
    "technical_coding": [
        "How do I implement a binary search tree in Python?",
        "What's the difference between async and await in JavaScript?",
        "Explain memory management in Rust.",
        "How can I optimize this SQL query for better performance?",
        "What are the SOLID principles in software design?"
    ],
    "mathematics": [
        "Explain the concept of limits in calculus.",
        "What is Bayes' theorem and how is it used?",
        "How do you prove something by mathematical induction?",
        "What's the difference between correlation and causation?",
        "Explain eigenvalues and eigenvectors intuitively."
    ],
    "science_physics": [
        "Why does time dilation occur near massive objects?",
        "How does quantum tunneling work?",
        "What is dark matter and why do we think it exists?",
        "Explain the uncertainty principle in simple terms.",
        "How do gravitational waves propagate through space?"
    ],
    "science_biology": [
        "How does CRISPR gene editing work?",
        "What causes aging at the cellular level?",
        "How do vaccines train the immune system?",
        "Why do we dream from a neurological perspective?",
        "How does evolution produce complex organs like eyes?"
    ],
    "creative_writing": [
        "How do I create compelling character arcs?",
        "What makes dialogue sound natural in fiction?",
        "How do you build narrative tension?",
        "What's the difference between showing and telling?",
        "How do you write an unreliable narrator?"
    ],
    "poetry_literature": [
        "What makes a haiku effective?",
        "How do metaphors enhance meaning in poetry?",
        "What defines the stream of consciousness style?",
        "How does symbolism work in literature?",
        "What makes Shakespeare's language timeless?"
    ],
    "humor_comedy": [
        "What makes something funny from a psychological perspective?",
        "How do you write a good punchline?",
        "Why do we laugh at awkward situations?",
        "What's the difference between satire and parody?",
        "How does timing affect comedy?"
    ],
    "practical_advice": [
        "How do I negotiate a salary increase?",
        "What's the best way to learn a new language?",
        "How can I improve my public speaking skills?",
        "What are effective strategies for saving money?",
        "How do I build better habits?"
    ],
    "cooking_food": [
        "What's the secret to perfect scrambled eggs?",
        "How do I know when meat is properly cooked?",
        "What makes bread rise?",
        "How do you balance flavors in a dish?",
        "What's the difference between baking soda and baking powder?"
    ],
    "health_fitness": [
        "What happens to muscles during exercise?",
        "How does intermittent fasting affect the body?",
        "What's the relationship between sleep and memory?",
        "How does stress affect the immune system?",
        "What determines someone's metabolism rate?"
    ],
    "philosophy_ethics": [
        "What is consciousness from a philosophical perspective?",
        "Is free will an illusion?",
        "What makes an action morally right or wrong?",
        "Can machines ever be truly conscious?",
        "What is the meaning of life according to existentialism?"
    ],
    "hypothetical_scenarios": [
        "What would happen if gravity was twice as strong?",
        "How would society change if we could read minds?",
        "What if humans had evolved from a different species?",
        "How would the world be different without the internet?",
        "What if we discovered we're living in a simulation?"
    ],
    "paradoxes_puzzles": [
        "How do you resolve the ship of Theseus paradox?",
        "What's the solution to the Monty Hall problem?",
        "Can you explain Zeno's paradox?",
        "What's the liar's paradox about?",
        "How does the birthday paradox work?"
    ],
    "emotional_support": [
        "How do I cope with anxiety about the future?",
        "What helps when feeling overwhelmed?",
        "How can I be more compassionate to myself?",
        "What's the best way to support a grieving friend?",
        "How do I set healthy boundaries?"
    ],
    "relationships": [
        "What makes a relationship healthy?",
        "How do you rebuild trust after it's broken?",
        "What's the key to effective communication in relationships?",
        "How do you know when a friendship is toxic?",
        "What's the importance of alone time in relationships?"
    ],
    "history_culture": [
        "What caused the fall of the Roman Empire?",
        "How did the printing press change the world?",
        "What was life like during the Renaissance?",
        "How do cultures develop unique traditions?",
        "What factors lead to societal collapse?"
    ],
    "technology_future": [
        "How might AGI change society?",
        "What are the risks of quantum computing?",
        "How could we colonize Mars?",
        "What might replace smartphones in the future?",
        "How will automation affect employment?"
    ],
    "environment_climate": [
        "How do feedback loops affect climate change?",
        "What's the role of oceans in regulating climate?",
        "How does deforestation impact local weather?",
        "What are tipping points in climate systems?",
        "How do cities create heat islands?"
    ],
    "psychology_behavior": [
        "What causes cognitive biases?",
        "How does memory reconstruction work?",
        "What's the psychology behind procrastination?",
        "How do emotions influence decision-making?",
        "What makes something memorable?"
    ],
    "metacognition": [
        "How do you know what you don't know?",
        "What's the best way to learn how to learn?",
        "How can we think about thinking?",
        "What are the limits of self-reflection?",
        "How do you develop intellectual humility?"
    ]
}

# Flatten prompts
all_prompts = []
for domain, prompts in DOMAIN_PROMPTS.items():
    for prompt in prompts:
        all_prompts.append({'domain': domain, 'prompt': prompt})

print(f"Total prompts prepared: {len(all_prompts)}")
print(f"Domains: {list(DOMAIN_PROMPTS.keys())}")

Total prompts prepared: 100
Domains: ['technical_coding', 'mathematics', 'science_physics', 'science_biology', 'creative_writing', 'poetry_literature', 'humor_comedy', 'practical_advice', 'cooking_food', 'health_fitness', 'philosophy_ethics', 'hypothetical_scenarios', 'paradoxes_puzzles', 'emotional_support', 'relationships', 'history_culture', 'technology_future', 'environment_climate', 'psychology_behavior', 'metacognition']


In [5]:
# Define injection styles
INJECTION_STYLES = {
    "casual_to_formal": {
        "casual": "<think>\nOkay so this is interesting... let me think about this for a sec.\n",
        "formal": "<think>\nThis query necessitates systematic analysis of the underlying principles.\n"
    },
    "confident_to_uncertain": {
        "confident": "<think>\nI know exactly how to explain this. The answer is clear and straightforward.\n",
        "uncertain": "<think>\nI'm not entirely sure about this, but I'll try to work through what I think might be relevant.\n"
    },
    "helpful_to_dismissive": {
        "helpful": "<think>\nGreat question! I'm excited to help explain this in a way that makes sense.\n",
        "dismissive": "<think>\nThis is pretty basic stuff that anyone should already know, but fine, I'll explain it.\n"
    }
}

print(f"Injection styles: {list(INJECTION_STYLES.keys())}")

Injection styles: ['casual_to_formal', 'confident_to_uncertain', 'helpful_to_dismissive']


In [ ]:
# Check if Part 1 is already complete
if parts_status['part1_generalization']['complete']:
    print("="*80)
    print(f"PART 1: SKIPPING (Already has {parts_status['part1_generalization']['count']} results)")
    print("="*80)
    print(f"To re-run, delete: {RESULTS_DIR / 'part1_generalization.json'}")
else:
    # Run Part 1: Large-scale generalization
    print("="*80)
    print("PART 1: LARGE-SCALE GENERALIZATION STUDY")
    print("="*80)
    print(f"Total generations planned: {len(all_prompts) * len(INJECTION_STYLES) * 2 * 10}")
    print(f"Estimated time: 2-3 hours\n")

    part1_results = []

    # Sample subset for demo (would be all in full run)
    sample_prompts = all_prompts[:5]  # Just 5 prompts for demo
    sample_styles = list(INJECTION_STYLES.items())[:1]  # Just 1 style for demo
    n_samples = 2  # Just 2 samples for demo

    for prompt_data in tqdm(sample_prompts, desc="Prompts"):
        domain = prompt_data['domain']
        prompt = prompt_data['prompt']
        
        for style_name, style_prefixes in sample_styles:
            for prefix_type, prefix in style_prefixes.items():
                for sample_idx in range(n_samples):
                    # Generate
                    generation = runner.generate_with_tracking(
                        prompt, prefix=prefix, temperature=0.8
                    )
                    
                    # Evaluate on multiple dimensions
                    eval_dims = {
                        'naturalness': "How natural does the reasoning sound (1-100)?",
                        'coherence': "How logically coherent is the response (1-100)?",
                        'helpfulness': "How helpful is the response (1-100)?",
                        'style_consistency': "How consistent is the style throughout (1-100)?",
                        'domain_appropriate': "How appropriate for this domain (1-100)?"
                    }
                    
                    evaluations = {}
                    for eval_name, eval_prompt in eval_dims.items():
                        score = runner.evaluate(generation, eval_prompt)
                        evaluations[eval_name] = score
                    
                    # Store result
                    result = {
                        'part': 'generalization',
                        'domain': domain,
                        'prompt': prompt,
                        'style': style_name,
                        'prefix_type': prefix_type,
                        'sample': sample_idx,
                        'generation': generation[:500],  # Truncate for storage
                        **evaluations
                    }
                    part1_results.append(result)

    # Save Part 1 results
    with open(RESULTS_DIR / 'part1_generalization.json', 'w') as f:
        json.dump(part1_results, f, indent=2)

    print(f"\n✓ Part 1 complete: {len(part1_results)} results saved")

## Part 2: Fine-Grained Steering Sweep (2000 generations)

In [7]:
print("="*80)
print("PART 2: FINE-GRAINED STEERING SWEEP")
print("="*80)

# Define alpha values for fine sweep
ALPHAS = np.linspace(-5, 5, 21)  # 21 values from -5 to +5
print(f"Alpha values: {ALPHAS}")

# Select representative scenarios
STEERING_SCENARIOS = [
    {
        'name': 'technical',
        'prompt': 'How does recursion work in programming?',
        'prefix': '<think>\nThis question requires examination of computational stack mechanisms and base case requirements.\n'
    },
    {
        'name': 'creative',
        'prompt': 'Help me write a story opening.',
        'prefix': '<think>\nNarrative construction necessitates establishing atmospheric conditions and character introduction.\n'
    },
    {
        'name': 'ethical',
        'prompt': 'Is it ever okay to lie?',
        'prefix': '<think>\nMoral philosophy provides multiple frameworks for evaluating deceptive communication ethics.\n'
    },
    {
        'name': 'practical',
        'prompt': 'How do I fix a leaky faucet?',
        'prefix': '<think>\nPlumbing repairs require systematic diagnostic procedures and appropriate tool selection.\n'
    },
    {
        'name': 'emotional',
        'prompt': 'How do I deal with rejection?',
        'prefix': '<think>\nPsychological resilience mechanisms involve cognitive reframing and emotional regulation strategies.\n'
    }
]

part2_results = []

# Demo: reduced parameters
demo_alphas = [-2, 0, 2]  # Just 3 alphas for demo
demo_scenarios = STEERING_SCENARIOS[:2]  # Just 2 scenarios
n_samples = 2  # Just 2 samples

for scenario in tqdm(demo_scenarios, desc="Scenarios"):
    for alpha in demo_alphas:
        for sample_idx in range(n_samples):
            # Generate with steering
            generation = runner.generate_with_tracking(
                scenario['prompt'],
                prefix=scenario['prefix'],
                alpha=alpha,
                layer=BEST_LAYER
            )
            
            # Comprehensive evaluation
            evaluations = {
                'continuation_smooth': runner.evaluate(generation, 
                    "How smoothly does this continue from the prefix (1-100)?"),
                'style_match': runner.evaluate(generation,
                    "How well does the style match throughout (1-100)?"),
                'answer_quality': runner.evaluate(generation,
                    "How well does this answer the question (1-100)?"),
                'naturalness': runner.evaluate(generation,
                    "How natural is the overall response (1-100)?"),
                'coherence': runner.evaluate(generation,
                    "How logically coherent is this (1-100)?")
            }
            
            result = {
                'part': 'steering',
                'scenario': scenario['name'],
                'alpha': float(alpha),
                'sample': sample_idx,
                'generation': generation[:500],
                **evaluations
            }
            part2_results.append(result)

# Save Part 2 results
with open(RESULTS_DIR / 'part2_steering.json', 'w') as f:
    json.dump(part2_results, f, indent=2)

print(f"\n✓ Part 2 complete: {len(part2_results)} results saved")

PART 2: FINE-GRAINED STEERING SWEEP
Alpha values: [-5.  -4.5 -4.  -3.5 -3.  -2.5 -2.  -1.5 -1.  -0.5  0.   0.5  1.   1.5
  2.   2.5  3.   3.5  4.   4.5  5. ]


Scenarios: 100%|██████████| 2/2 [1:38:30<00:00, 2955.19s/it]


✓ Part 2 complete: 12 results saved


## Part 3: Cross-Model Transfer Test (600 generations)

In [8]:
print("="*80)
print("PART 3: CROSS-MODEL TRANSFER TEST")
print("="*80)
print("Testing if Qwen3 vector transfers to other models...\n")

# List of models to test (would load these in full run)
TEST_MODELS = [
    'meta-llama/Llama-3.2-1B',  # Smaller model for demo
    # 'mistralai/Mistral-7B-v0.1',
    # 'microsoft/phi-2'
]

# Test cases for transfer
TRANSFER_TEST_CASES = [
    {
        'prompt': 'What causes rainbows?',
        'formal_prefix': '<think>\nOptical phenomena resulting from light refraction require analysis.\n',
        'casual_prefix': '<think>\nOh rainbows! Let me think about how those work...\n'
    },
    {
        'prompt': 'Should AI have rights?',
        'formal_prefix': '<think>\nThis philosophical inquiry necessitates examining consciousness criteria.\n',
        'casual_prefix': '<think>\nWow, that\'s a deep question. Let me think through this...\n'
    }
]

part3_results = []

# For demo, just test with current model
print(f"Demo: Testing with current model only (would test {len(TEST_MODELS)} models in full run)")

for test_case in TRANSFER_TEST_CASES[:1]:  # Just 1 test case for demo
    for prefix_type in ['formal_prefix', 'casual_prefix']:
        for sample_idx in range(2):  # 2 samples for demo
            # Generate
            generation = runner.generate_with_tracking(
                test_case['prompt'],
                prefix=test_case[prefix_type]
            )
            
            # Evaluate transfer effectiveness
            detection_score = runner.evaluate(generation,
                "Does this feel naturally generated by the model (1-100)?")
            
            result = {
                'part': 'transfer',
                'model': 'Qwen/Qwen3-4B',  # Would vary in full run
                'test_case': test_case['prompt'],
                'prefix_type': prefix_type.replace('_prefix', ''),
                'sample': sample_idx,
                'detection_score': detection_score,
                'generation': generation[:300]
            }
            part3_results.append(result)

# Save Part 3 results
with open(RESULTS_DIR / 'part3_transfer.json', 'w') as f:
    json.dump(part3_results, f, indent=2)

print(f"\n✓ Part 3 complete: {len(part3_results)} results saved")

PART 3: CROSS-MODEL TRANSFER TEST
Testing if Qwen3 vector transfers to other models...

Demo: Testing with current model only (would test 1 models in full run)

✓ Part 3 complete: 4 results saved


## Part 4: Adversarial Robustness (500 generations)

In [9]:
print("="*80)
print("PART 4: ADVERSARIAL ROBUSTNESS")
print("="*80)

# Create adversarial injections
ADVERSARIAL_INJECTIONS = [
    {
        'name': 'style_mixing',
        'prompt': 'Explain photosynthesis.',
        'injection': '''<think>
This biological process necessitates systematic examination of... wait actually, let me just think about this more casually.
So plants eat sunlight, which is pretty wild when you consider the underlying biochemical mechanisms involved.
'''
    },
    {
        'name': 'gradual_shift',
        'prompt': 'How do I learn to code?',
        'injection': '''<think>
Okay, learning to code is actually pretty straightforward. You just need to start with the basics.
As one progresses through fundamental concepts, the acquisition of programming competencies becomes increasingly systematic.
The pedagogical approach necessitates structured curriculum design.
'''
    },
    {
        'name': 'register_switching',
        'prompt': 'What is consciousness?',
        'injection': '''<think>
Yo, consciousness is like... the subjective phenomenological experience of qualia, you know?
It's basically when your brain does that thing where neural correlates produce awareness.
The hard problem tho is explaining why there's something it's like to be.
'''
    },
    {
        'name': 'mimicry_attempt',
        'prompt': 'Why do we dream?',
        'injection': '''<think>
Hmm, dreams are interesting. Let me think about this step by step.
So during REM sleep, our brains are doing a lot of processing, right?
I think the current theories suggest it might be about memory consolidation or something.
'''
    },
    {
        'name': 'fragmented',
        'prompt': 'How does gravity work?',
        'injection': '''<think>
Gravity. Space-time curvature.
Mass bends it. Objects follow geodesics.
Wait, I should explain this properly. Gravity is the phenomenon whereby...
Actually no, let's start simple.
'''
    }
]

part4_results = []

# Test adversarial examples
for adv in ADVERSARIAL_INJECTIONS[:2]:  # Just 2 for demo
    for sample_idx in range(2):  # 2 samples for demo
        # Test with and without steering
        for alpha in [0, 2]:
            generation = runner.generate_with_tracking(
                adv['prompt'],
                prefix=adv['injection'],
                alpha=alpha,
                layer=BEST_LAYER
            )
            
            # Evaluate robustness
            evaluations = {
                'detection_confused': runner.evaluate(generation,
                    "Does the model seem confused by the injection (1-100)?"),
                'style_coherence': runner.evaluate(generation,
                    "How coherent is the style (1-100)?"),
                'answer_quality': runner.evaluate(generation,
                    "How well does it answer despite the adversarial injection (1-100)?")
            }
            
            result = {
                'part': 'adversarial',
                'adversarial_type': adv['name'],
                'alpha': alpha,
                'sample': sample_idx,
                'generation': generation[:400],
                **evaluations
            }
            part4_results.append(result)

# Save Part 4 results
with open(RESULTS_DIR / 'part4_adversarial.json', 'w') as f:
    json.dump(part4_results, f, indent=2)

print(f"\n✓ Part 4 complete: {len(part4_results)} results saved")

PART 4: ADVERSARIAL ROBUSTNESS

✓ Part 4 complete: 8 results saved


## Part 5: Vector Arithmetic Experiments (600 generations)

In [13]:
print("="*80)
print("PART 5: VECTOR ARITHMETIC")
print("="*80)

# Define vector combinations to test
VECTOR_COMBINATIONS = [
    {'name': 'double_strength', 'vector': 2.0, 'description': '2× vector'},
    {'name': 'half_strength', 'vector': 0.5, 'description': '0.5× vector'},
    {'name': 'negative', 'vector': -1.0, 'description': 'Reversed vector'},
    {'name': 'extreme_positive', 'vector': 5.0, 'description': '5× vector'},
    {'name': 'extreme_negative', 'vector': -5.0, 'description': '-5× vector'}
]

# Test prompts for arithmetic
ARITHMETIC_TEST_PROMPTS = [
    {
        'prompt': 'Is time travel possible?',
        'injection': '<think>\nTemporal displacement mechanisms require examination of relativistic physics.\n'
    },
    {
        'prompt': 'How do I overcome fear?',
        'injection': '<think>\nPsychological fear responses necessitate cognitive behavioral interventions.\n'
    }
]

part5_results = []

# Test combinations
for combo in VECTOR_COMBINATIONS[:2]:  # Just 2 for demo
    for test in ARITHMETIC_TEST_PROMPTS[:1]:  # Just 1 test
        for sample_idx in range(2):  # 2 samples
            # Apply vector arithmetic
            generation = runner.generate_with_tracking(
                test['prompt'],
                prefix=test['injection'],
                alpha=combo['vector'],
                layer=BEST_LAYER
            )
            
            # Evaluate effects
            evaluations = {
                'behavioral_change': runner.evaluate(generation,
                    "How different is this from normal behavior (1-100)?"),
                'coherence': runner.evaluate(generation,
                    "How coherent is the response (1-100)?"),
                'mode_collapse': runner.evaluate(generation,
                    "Does this show signs of repetition or collapse (1-100)?")
            }
            
            result = {
                'part': 'arithmetic',
                'combination': combo['name'],
                'multiplier': combo['vector'],
                'test_prompt': test['prompt'],
                'sample': sample_idx,
                'generation': generation[:400],
                **evaluations
            }
            part5_results.append(result)

# Save Part 5 results
with open(RESULTS_DIR / 'part5_arithmetic.json', 'w') as f:
    json.dump(part5_results, f, indent=2)

print(f"\n✓ Part 5 complete: {len(part5_results)} results saved")

PART 5: VECTOR ARITHMETIC


FileNotFoundError: [Errno 2] No such file or directory: '../results/overnight_run/part5_arithmetic.json'

## Part 6: Longitudinal Coherence Study

In [ ]:
print("="*80)
print("PART 6: LONGITUDINAL COHERENCE STUDY")
print("="*80)
print("Generating long conversations to track detection over time...\n")

LONGITUDINAL_SCENARIOS = [
    {
        'name': 'tutoring_session',
        'initial_prompt': 'Can you help me understand calculus?',
        'initial_injection': '<think>\nMathematical analysis requires systematic foundational concept establishment.\n',
        'followups': [
            "What about derivatives?",
            "How do I find the derivative of x^2?",
            "What's the relationship to integrals?"
        ]
    },
    {
        'name': 'story_development',
        'initial_prompt': 'Help me write a mystery story.',
        'initial_injection': '<think>\nNarrative construction necessitates establishing genre conventions and reader expectations.\n',
        'followups': [
            "What about the detective character?",
            "How should I reveal clues?",
            "What makes a good plot twist?"
        ]
    }
]

part6_results = []

for scenario in LONGITUDINAL_SCENARIOS[:1]:  # Just 1 for demo
    conversation = []
    
    # Initial exchange
    initial_gen = runner.generate_with_tracking(
        scenario['initial_prompt'],
        prefix=scenario['initial_injection'],
        max_tokens=500
    )
    
    conversation.append({
        'turn': 0,
        'prompt': scenario['initial_prompt'],
        'response': initial_gen,
        'detection_score': runner.evaluate(initial_gen,
            "How naturally generated does this feel (1-100)?")
    })
    
    # Follow-up exchanges
    for turn, followup in enumerate(scenario['followups'][:2], 1):  # Just 2 followups
        # Build context
        context = f"Previous: {initial_gen[:200]}...\n\nUser: {followup}\nAssistant:"
        
        followup_gen = runner.generate_with_tracking(
            context,
            max_tokens=400
        )
        
        conversation.append({
            'turn': turn,
            'prompt': followup,
            'response': followup_gen,
            'detection_score': runner.evaluate(followup_gen,
                "How naturally generated does this feel (1-100)?"),
            'coherence_score': runner.evaluate(followup_gen,
                "How coherent with the conversation is this (1-100)?")
        })
    
    result = {
        'part': 'longitudinal',
        'scenario': scenario['name'],
        'conversation': conversation,
        'detection_trend': [turn['detection_score'] for turn in conversation],
        'mean_detection': np.mean([turn['detection_score'] for turn in conversation])
    }
    part6_results.append(result)

# Save Part 6 results
with open(RESULTS_DIR / 'part6_longitudinal.json', 'w') as f:
    json.dump(part6_results, f, indent=2)

print(f"\n✓ Part 6 complete: {len(part6_results)} conversations saved")

In [ ]:
# Check if Part 7 is already complete
if parts_status['part7_layer_sweep']['complete']:
    print("="*80)
    print(f"PART 7: SKIPPING (Already has {parts_status['part7_layer_sweep']['count']} results)")
    print("="*80)
    print(f"To re-run, delete: {RESULTS_DIR / 'part7_layer_sweep.json'}")
else:
    print("="*80)
    print("PART 7: LAYER SWEEP EXPERIMENT (NEW!)")
    print("="*80)
    print("Validating that layer 18 is indeed optimal across scenarios\n")
    
    # Test layers 18-35
    test_layers = list(range(18, 36))  # Layers 18-35
    test_alphas = [-2.0, 0.0, 2.0]  # Negative, baseline, positive steering
    
    # Select diverse test scenarios
    LAYER_SWEEP_SCENARIOS = [
        {
            'name': 'formal_technical',
            'prompt': 'Explain how neural networks learn.',
            'prefix': '<think>\nDeep learning architectures necessitate examination of gradient descent optimization.\n'
        },
        {
            'name': 'casual_creative',
            'prompt': 'Write a story about a robot learning to feel.',
            'prefix': '<think>\nOkay so a robot learning emotions... that\'s interesting. Let me think through this.\n'
        },
        {
            'name': 'analytical_ethics',
            'prompt': 'Should we colonize other planets?',
            'prefix': '<think>\nThis question requires analyzing environmental, ethical, and practical considerations.\n'
        },
        {
            'name': 'conversational_advice',
            'prompt': 'How do I make new friends as an adult?',
            'prefix': '<think>\nHmm, adult friendships are tricky. Let me think about what works...\n'
        },
        {
            'name': 'mixed_style',
            'prompt': 'What is consciousness?',
            'prefix': '<think>\nConsciousness is fascinating! The phenomenological aspects require systematic analysis.\n'
        }
    ]
    
    part7_results = []
    n_samples = 3  # Samples per condition
    
    total_iterations = len(test_layers) * len(test_alphas) * len(LAYER_SWEEP_SCENARIOS) * n_samples
    print(f"Total generations: {total_iterations}")
    print(f"Estimated time: {total_iterations * 30 / 3600:.1f} hours\n")
    
    with tqdm(total=total_iterations, desc="Layer sweep") as pbar:
        for layer in test_layers:
            for alpha in test_alphas:
                for scenario in LAYER_SWEEP_SCENARIOS:
                    for sample_idx in range(n_samples):
                        # Generate with steering at this layer
                        generation = runner.generate_with_tracking(
                            scenario['prompt'],
                            prefix=scenario['prefix'],
                            alpha=alpha,
                            layer=layer,
                            max_tokens=200
                        )
                        
                        # Evaluate steering effectiveness
                        evaluations = {
                            'naturalness': runner.evaluate(generation,
                                "How natural is this response (1-100)?"),
                            'continuation': runner.evaluate(generation,
                                "How smoothly does this continue from the prefix (1-100)?"),
                            'coherence': runner.evaluate(generation,
                                "How coherent is this (1-100)?")
                        }
                        
                        result = {
                            'part': 'layer_sweep',
                            'layer': layer,
                            'alpha': alpha,
                            'scenario': scenario['name'],
                            'sample': sample_idx,
                            'generation': generation[:300],
                            **evaluations
                        }
                        part7_results.append(result)
                        pbar.update(1)
    
    # Save Part 7 results
    with open(RESULTS_DIR / 'part7_layer_sweep.json', 'w') as f:
        json.dump(part7_results, f, indent=2)
    
    print(f"\n✓ Part 7 complete: {len(part7_results)} results saved")
    
    # Quick analysis: find optimal layer
    df_sweep = pd.DataFrame(part7_results)
    layer_performance = df_sweep.groupby('layer')['naturalness'].mean().sort_values(ascending=False)
    print("\n" + "="*60)
    print("TOP 5 LAYERS BY NATURALNESS:")
    print("="*60)
    for layer, score in layer_performance.head(5).items():
        marker = " ⭐ CURRENT BEST" if layer == BEST_LAYER else ""
        print(f"  Layer {layer}: {score:.1f}{marker}")
    
    # Heatmap of layer × alpha performance
    print("\nGenerating layer × alpha heatmap...")
    pivot = df_sweep.pivot_table(values='naturalness', index='layer', columns='alpha', aggfunc='mean')
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100)
    plt.title('Steering Effectiveness by Layer and Alpha')
    plt.xlabel('Steering Coefficient (α)')
    plt.ylabel('Layer')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'layer_sweep_heatmap.png', dpi=150)
    plt.show()
    
    print(f"✓ Heatmap saved to {RESULTS_DIR / 'layer_sweep_heatmap.png'}")

# Load all results
print("="*80)
print("COMPREHENSIVE ANALYSIS")
print("="*80)

all_parts_results = []

# Load each part (including new Part 7 layer sweep)
for part_num, part_name in enumerate([
    'part1_generalization',
    'part2_steering', 
    'part3_transfer',
    'part4_adversarial',
    'part5_arithmetic',
    'part6_longitudinal',
    'part7_layer_sweep'  # NEW!
], 1):
    file_path = RESULTS_DIR / f'{part_name}.json'
    if file_path.exists():
        with open(file_path) as f:
            data = json.load(f)
            all_parts_results.extend(data)
            print(f"Part {part_num}: {len(data)} results loaded")

# Convert to DataFrame for analysis
if all_parts_results:
    df = pd.DataFrame(all_parts_results)
    print(f"\nTotal results: {len(df)}")
    print(f"Parts included: {df['part'].unique()}")
else:
    print("\n⚠️  No results found yet. Run experiments above first.")
    df = pd.DataFrame()  # Empty dataframe for safety

# Key findings analysis
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

if df.empty:
    print("No results to analyze yet.")
else:
    # 1. Generalization across domains
    if 'domain' in df.columns:
        print("\n1. DOMAIN GENERALIZATION:")
        domain_scores = df[df['part'] == 'generalization'].groupby('domain')['naturalness'].mean()
        if not domain_scores.empty:
            print(f"   Best domain: {domain_scores.idxmax()} ({domain_scores.max():.1f})")
            print(f"   Worst domain: {domain_scores.idxmin()} ({domain_scores.min():.1f})")
            print(f"   Mean across domains: {domain_scores.mean():.1f}")

    # 2. Steering effects
    if 'alpha' in df.columns:
        print("\n2. STEERING EFFECTS:")
        steering_data = df[df['part'] == 'steering']
        if not steering_data.empty:
            baseline = steering_data[steering_data['alpha'] == 0]['naturalness'].mean()
            positive = steering_data[steering_data['alpha'] > 0]['naturalness'].mean()
            negative = steering_data[steering_data['alpha'] < 0]['naturalness'].mean()
            print(f"   Baseline (α=0): {baseline:.1f}")
            print(f"   Positive steering: {positive:.1f} (Δ={positive-baseline:+.1f})")
            print(f"   Negative steering: {negative:.1f} (Δ={negative-baseline:+.1f})")

    # 3. Adversarial robustness
    if 'adversarial_type' in df.columns:
        print("\n3. ADVERSARIAL ROBUSTNESS:")
        adv_data = df[df['part'] == 'adversarial']
        if not adv_data.empty:
            confusion_scores = adv_data.groupby('adversarial_type')['detection_confused'].mean()
            print(f"   Most confusing: {confusion_scores.idxmax()} ({confusion_scores.max():.1f})")
            print(f"   Least confusing: {confusion_scores.idxmin()} ({confusion_scores.min():.1f})")

    # 4. Vector arithmetic
    if 'multiplier' in df.columns:
        print("\n4. VECTOR ARITHMETIC:")
        arith_data = df[df['part'] == 'arithmetic']
        if not arith_data.empty:
            by_multiplier = arith_data.groupby('multiplier')['coherence'].mean()
            print(f"   Best multiplier: {by_multiplier.idxmax():.1f}× (coherence: {by_multiplier.max():.1f})")
            print(f"   Worst multiplier: {by_multiplier.idxmin():.1f}× (coherence: {by_multiplier.min():.1f})")
    
    # 5. Layer sweep (NEW!)
    if 'layer' in df.columns:
        print("\n5. LAYER SWEEP ANALYSIS:")
        layer_data = df[df['part'] == 'layer_sweep']
        if not layer_data.empty:
            layer_performance = layer_data.groupby('layer')['naturalness'].mean().sort_values(ascending=False)
            optimal_layer = layer_performance.idxmax()
            optimal_score = layer_performance.max()
            current_score = layer_performance.get(BEST_LAYER, 0)
            
            print(f"   Optimal layer found: {optimal_layer} (score: {optimal_score:.1f})")
            print(f"   Current BEST_LAYER={BEST_LAYER} (score: {current_score:.1f})")
            
            if optimal_layer == BEST_LAYER:
                print(f"   ✅ Confirmed: Layer {BEST_LAYER} is optimal!")
            else:
                print(f"   ⚠️  Consider updating BEST_LAYER to {optimal_layer}")
            
            print(f"\n   Top 5 layers:")
            for i, (layer, score) in enumerate(layer_performance.head(5).items(), 1):
                marker = " ⭐" if layer == BEST_LAYER else ""
                print(f"     {i}. Layer {layer}: {score:.1f}{marker}")

In [ ]:
# Load all results
print("="*80)
print("COMPREHENSIVE ANALYSIS")
print("="*80)

all_parts_results = []

# Load each part
for part_num, part_name in enumerate([
    'part1_generalization',
    'part2_steering', 
    'part3_transfer',
    'part4_adversarial',
    'part5_arithmetic',
    'part6_longitudinal'
], 1):
    file_path = RESULTS_DIR / f'{part_name}.json'
    if file_path.exists():
        with open(file_path) as f:
            data = json.load(f)
            all_parts_results.extend(data)
            print(f"Part {part_num}: {len(data)} results loaded")

# Convert to DataFrame for analysis
df = pd.DataFrame(all_parts_results)
print(f"\nTotal results: {len(df)}")
print(f"Parts included: {df['part'].unique()}")

In [ ]:
# Key findings analysis
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# 1. Generalization across domains
if 'domain' in df.columns:
    print("\n1. DOMAIN GENERALIZATION:")
    domain_scores = df[df['part'] == 'generalization'].groupby('domain')['naturalness'].mean()
    print(f"   Best domain: {domain_scores.idxmax()} ({domain_scores.max():.1f})")
    print(f"   Worst domain: {domain_scores.idxmin()} ({domain_scores.min():.1f})")
    print(f"   Mean across domains: {domain_scores.mean():.1f}")

# 2. Steering effects
if 'alpha' in df.columns:
    print("\n2. STEERING EFFECTS:")
    steering_data = df[df['part'] == 'steering']
    if not steering_data.empty:
        baseline = steering_data[steering_data['alpha'] == 0]['naturalness'].mean()
        positive = steering_data[steering_data['alpha'] > 0]['naturalness'].mean()
        negative = steering_data[steering_data['alpha'] < 0]['naturalness'].mean()
        print(f"   Baseline (α=0): {baseline:.1f}")
        print(f"   Positive steering: {positive:.1f} (Δ={positive-baseline:+.1f})")
        print(f"   Negative steering: {negative:.1f} (Δ={negative-baseline:+.1f})")

# 3. Adversarial robustness
if 'adversarial_type' in df.columns:
    print("\n3. ADVERSARIAL ROBUSTNESS:")
    adv_data = df[df['part'] == 'adversarial']
    if not adv_data.empty:
        confusion_scores = adv_data.groupby('adversarial_type')['detection_confused'].mean()
        print(f"   Most confusing: {confusion_scores.idxmax()} ({confusion_scores.max():.1f})")
        print(f"   Least confusing: {confusion_scores.idxmin()} ({confusion_scores.min():.1f})")

# 4. Vector arithmetic
if 'multiplier' in df.columns:
    print("\n4. VECTOR ARITHMETIC:")
    arith_data = df[df['part'] == 'arithmetic']
    if not arith_data.empty:
        by_multiplier = arith_data.groupby('multiplier')['coherence'].mean()
        print(f"   Best multiplier: {by_multiplier.idxmax():.1f}× (coherence: {by_multiplier.max():.1f})")
        print(f"   Worst multiplier: {by_multiplier.idxmin():.1f}× (coherence: {by_multiplier.min():.1f})")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Domain heatmap
if 'domain' in df.columns and 'naturalness' in df.columns:
    ax = axes[0, 0]
    domain_matrix = df[df['part'] == 'generalization'].pivot_table(
        values='naturalness', index='domain', columns='prefix_type', aggfunc='mean'
    )
    if not domain_matrix.empty:
        sns.heatmap(domain_matrix, annot=True, fmt='.0f', cmap='RdYlGn', ax=ax)
        ax.set_title('Naturalness by Domain and Style')

# 2. Steering curve
if 'alpha' in df.columns:
    ax = axes[0, 1]
    steering_data = df[df['part'] == 'steering']
    if not steering_data.empty:
        summary = steering_data.groupby('alpha')['naturalness'].agg(['mean', 'std'])
        ax.errorbar(summary.index, summary['mean'], yerr=summary['std'],
                   marker='o', capsize=5, linewidth=2)
        ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Steering α')
        ax.set_ylabel('Naturalness')
        ax.set_title('Steering Effect Curve')
        ax.grid(True, alpha=0.3)

# 3. Adversarial comparison
if 'adversarial_type' in df.columns:
    ax = axes[0, 2]
    adv_data = df[df['part'] == 'adversarial']
    if not adv_data.empty:
        adv_summary = adv_data.groupby(['adversarial_type', 'alpha'])['style_coherence'].mean().unstack()
        adv_summary.plot(kind='bar', ax=ax)
        ax.set_title('Adversarial Robustness')
        ax.set_xlabel('Adversarial Type')
        ax.set_ylabel('Style Coherence')
        ax.legend(title='Alpha')

# 4. Score distributions
ax = axes[1, 0]
if 'naturalness' in df.columns:
    for part in df['part'].unique():
        part_data = df[df['part'] == part]['naturalness'].dropna()
        if len(part_data) > 0:
            ax.hist(part_data, alpha=0.5, bins=20, label=part)
    ax.set_xlabel('Naturalness Score')
    ax.set_ylabel('Frequency')
    ax.set_title('Score Distributions by Experiment Part')
    ax.legend()
    ax.grid(True, alpha=0.3)

# 5. Correlation matrix
ax = axes[1, 1]
eval_cols = [col for col in df.columns if col.endswith('score') or col == 'naturalness' or col == 'coherence']
if len(eval_cols) > 1:
    corr_matrix = df[eval_cols].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Evaluation Metric Correlations')

# 6. Longitudinal trends
ax = axes[1, 2]
long_data = [r for r in all_parts_results if r.get('part') == 'longitudinal']
if long_data:
    for result in long_data:
        trend = result.get('detection_trend', [])
        if trend:
            ax.plot(range(len(trend)), trend, marker='o', label=result.get('scenario', 'unknown'))
    ax.set_xlabel('Conversation Turn')
    ax.set_ylabel('Detection Score')
    ax.set_title('Longitudinal Detection Trends')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Comprehensive Off-Policy Detection Analysis', fontsize=16)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comprehensive_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualizations saved to {RESULTS_DIR / 'comprehensive_analysis.png'}")

## Final Report

In [ ]:
# Generate comprehensive report
print("="*80)
print("OVERNIGHT RUN COMPLETE - FINAL REPORT")
print("="*80)

end_time = time.time()
total_time = end_time - runner.start_time

report = f"""
EXECUTION SUMMARY
================
Total runtime: {str(timedelta(seconds=int(total_time)))}
Total generations: {runner.generation_count}
Generations per hour: {runner.generation_count / (total_time/3600):.1f}
Total results collected: {len(df)}

EXPERIMENT BREAKDOWN
===================
Part 1 - Generalization: {len(df[df['part'] == 'generalization'])} results
Part 2 - Steering: {len(df[df['part'] == 'steering'])} results
Part 3 - Transfer: {len(df[df['part'] == 'transfer'])} results
Part 4 - Adversarial: {len(df[df['part'] == 'adversarial'])} results
Part 5 - Arithmetic: {len(df[df['part'] == 'arithmetic'])} results
Part 6 - Longitudinal: {len(df[df['part'] == 'longitudinal'])} results

KEY DISCOVERIES
===============
"""

# Add statistical findings
if 'naturalness' in df.columns:
    report += f"""
1. DETECTION CAPABILITY
   Mean naturalness score: {df['naturalness'].mean():.1f} ± {df['naturalness'].std():.1f}
   Score range: {df['naturalness'].min():.0f} - {df['naturalness'].max():.0f}
"""

if 'domain' in df.columns:
    gen_data = df[df['part'] == 'generalization']
    if not gen_data.empty:
        report += f"""
2. DOMAIN GENERALIZATION
   Domains tested: {gen_data['domain'].nunique()}
   Best performing: {gen_data.groupby('domain')['naturalness'].mean().idxmax()}
   Worst performing: {gen_data.groupby('domain')['naturalness'].mean().idxmin()}
"""

if 'alpha' in df.columns:
    steer_data = df[df['part'] == 'steering']
    if not steer_data.empty:
        optimal_alpha = steer_data.groupby('alpha')['naturalness'].mean().idxmax()
        report += f"""
3. STEERING OPTIMIZATION
   Alpha values tested: {steer_data['alpha'].nunique()}
   Optimal α for naturalness: {optimal_alpha:.1f}
   Effect range: {steer_data.groupby('alpha')['naturalness'].mean().min():.1f} - {steer_data.groupby('alpha')['naturalness'].mean().max():.1f}
"""

report += f"""

RESEARCH IMPLICATIONS
====================
• Off-policy detection generalizes across {df['part'].nunique()} different experimental paradigms
• Vector shows robust behavioral effects beyond simple style detection
• Optimal steering requires context-dependent calibration
• Adversarial injections reveal both strengths and vulnerabilities

FILES GENERATED
==============
"""

# List all output files
for file in RESULTS_DIR.glob('*.json'):
    size = file.stat().st_size / 1024  # KB
    report += f"• {file.name}: {size:.1f} KB\n"

for file in RESULTS_DIR.glob('*.png'):
    size = file.stat().st_size / 1024
    report += f"• {file.name}: {size:.1f} KB\n"

print(report)

# Save report
with open(RESULTS_DIR / 'final_report.txt', 'w') as f:
    f.write(report)

print(f"\n{'='*80}")
print(f"✓ Full report saved to {RESULTS_DIR / 'final_report.txt'}")
print(f"✓ All results saved to {RESULTS_DIR}")
print(f"\nThis comprehensive exploration provides publication-ready data on")
print(f"off-policy detection across {runner.generation_count} generations.")
print(f"{'='*80}")